# Seeding PM Cards – Bronze → Gold with Vector Search

This notebook ingests Preventive Maintenance (PM) cards for welding robots into a medallion architecture:

- **Bronze layer** (`demo.bronze_layer.pm_cards`): Raw PM card data ingested from seed SQL.
- **Gold layer** (`demo.gold_layer.pm_cards`): Enriched with `combined_text` (cause+symptom+action) and pre-computed `embedding` (`ARRAY<FLOAT>`) for semantic search.
- **Vector Search**: A Databricks Direct Vector Index on the gold table embedding column enables semantic similarity queries.

**Source of truth:** `anomaly-detection/data/sql/seed_databricks_pm_cards.sql` (45 rows, 4 robots + shared fleet).

**Requirements:** Databricks Runtime ML (for `sentence-transformers`), SparkSession (`spark`), databricks-sdk installed on cluster.

In [ ]:
# %% Setup: Imports & Configuration
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType, StructField, StructType, StringType
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    VectorSearchIndexType,
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
)

# ---------------------------------------------------------------------------
# Configuration — update catalog if your Unity Catalog differs
# ---------------------------------------------------------------------------
CATALOG          = "demo"
BRONZE_SCHEMA    = "bronze_layer"
GOLD_SCHEMA      = "gold_layer"
BRONZE_TABLE     = f"{CATALOG}.{BRONZE_SCHEMA}.pm_cards"
GOLD_TABLE       = f"{CATALOG}.{GOLD_SCHEMA}.pm_cards"

VS_ENDPOINT_NAME = "pm-cards-vs-endpoint"
VS_INDEX_NAME    = f"{CATALOG}.{GOLD_SCHEMA}.pm_cards_vs_index"

# Embedding model settings
#   - sentence-transformers/all-MiniLM-L6-v2 for local pre-computed embeddings
#   - databricks-bge-large-en for Vector Search model endpoint (Databricks-managed)
EMBEDDING_MODEL     = "all-MiniLM-L6-v2"   # 384-dim, lightweight
EMBEDDING_DIM       = 384
VS_EMBEDDING_MODEL  = "databricks-bge-large-en"  # Databricks foundation model

print(f"Bronze table          : {BRONZE_TABLE}")
print(f"Gold table            : {GOLD_TABLE}")
print(f"VS endpoint           : {VS_ENDPOINT_NAME}")
print(f"VS index              : {VS_INDEX_NAME}")
print(f"Pre-compute model     : {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")
print(f"VS model endpoint     : {VS_EMBEDDING_MODEL}")
print()
print(f"Spark version         : {spark.version}")

---
## Phase 1: Bronze Layer — Raw PM Cards Ingestion

In [ ]:
# %% Bronze: Create schema and table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    card_id     STRING,
    sensor      STRING,
    robot_code  STRING,
    cause       STRING,
    symptom     STRING,
    action      STRING
)
USING DELTA
LOCATION ?
""".replace("?", "''"))  # managed table

print(f"✓ Bronze table ensured: {BRONZE_TABLE}")
print(f"  Columns: card_id, sensor, robot_code, cause, symptom, action")

In [ ]:
# %% Bronze: Seed 45 PM card rows (idempotent — delete then insert)
print("Clearing existing data for idempotent re-run...")
spark.sql(f"DELETE FROM {BRONZE_TABLE}")

print("Inserting 45 PM card rows...")

# ── R-00: wire_feed_speed (4) + wire_temperature (3) + welding_current (4) = 11
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('PM-R00-WFS-001', 'wire_feed_speed', 'R-00',
 'Wire jam at drive rolls due to debris buildup',
 'Intermittent feed speed dropout, wire stutter during travel',
 'Clean drive rolls, replace feed liners, verify wire tension setpoint'),
('PM-R00-WFS-002', 'wire_feed_speed', 'R-00',
 'Worn feed roller grooves causing slip',
 'Gradual decline in wire_feed_speed, inconsistent bead deposition',
 'Replace feed rollers, calibrate feed tension, verify wire diameter match'),
('PM-R00-WFS-003', 'wire_feed_speed', 'R-00',
 'Encoder coupling degradation on feed motor',
 'Speed oscillation ±5% around setpoint, position error accumulating',
 'Replace encoder coupling, align shaft, verify feedback signal'),
('PM-R00-WFS-004', 'wire_feed_speed', 'R-00',
 'Wire spool brake binding',
 'Sudden feed stops at spool rotation start, jerky wire pull',
 'Clean and lubricate spool brake mechanism, replace brake pad if worn'),
('PM-R00-WT-001', 'wire_temperature', 'R-00',
 'Coolant line partial blockage in torch neck',
 'Wire temperature drift upward during continuous welding cycles',
 'Flush coolant lines, check flow rate at torch neck, replace coolant filter'),
('PM-R00-WT-002', 'wire_temperature', 'R-00',
 'Ambient temperature rise from HVAC zone imbalance',
 'Baseline wire temperature shift matching production hall temperature',
 'Balance HVAC zones, add local air circulation at welding station R-00'),
('PM-R00-WT-003', 'wire_temperature', 'R-00',
 'Insulation breakdown on welding cable near robot wrist',
 'Temperature excursions during high-current cycles, intermittent cable heating',
 'Inspect cable jacket integrity from cabinet to wrist, replace if cracked or melted'),
('PM-R00-WC-001', 'welding_current', 'R-00',
 'Contact tip bore wear from extended use',
 'Current readings show high-frequency oscillation, unstable arc',
 'Replace contact tip, verify bore diameter matches wire size'),
('PM-R00-WC-002', 'welding_current', 'R-00',
 'Poor ground connection at table clamp',
 'Intermittent current drops, arc restart difficulty',
 'Clean ground clamp contact surface, tighten ground cable lug, measure resistance < 5 mOhm'),
('PM-R00-WC-003', 'welding_current', 'R-00',
 'Power supply capacitor bank ageing',
 'Current surge events during peak demand cycles, ripple on waveform',
 'Test capacitance bank, schedule replacement if below 80% rated capacity'),
('PM-R00-WC-004', 'welding_current', 'R-00',
 'Welding cable connector oxidation at intermediate junction',
 'Current reading spikes during robot arm movement, intermittent contact',
 'Disassemble and clean cable connector, apply conductive grease, re-torque to spec')
""")
print("  ✓ R-00: 11 cards inserted (4 wire_feed_speed + 3 wire_temperature + 4 welding_current)")

In [ ]:
# %% Bronze: Insert R-01 cards (10 total)
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('PM-R01-WFS-001', 'wire_feed_speed', 'R-01',
 'Wire straightener tension too high causing wire breakage',
 'Feed speed drops to zero, followed by wire break event',
 'Reduce straightener tension, verify wire cast and helix within spec'),
('PM-R01-WFS-002', 'wire_feed_speed', 'R-01',
 'Drive roll pressure misadjustment',
 'Inconsistent feed speed tracking, wire slipping at high speeds',
 'Check and adjust drive roll pressure using pressure gauge, replace worn rolls'),
('PM-R01-WFS-003', 'wire_feed_speed', 'R-01',
 'Cable drag chain wear at R-01 wrist joint',
 'Intermittent dropout correlated with robot pose change at wrist',
 'Inspect drag chain for broken links or sharp bends, replace section if damaged'),
('PM-R01-WFS-004', 'wire_feed_speed', 'R-01',
 'Wire inlet guide tube blockage from dust',
 'Gradual feed resistance increase, motor current rise without speed change',
 'Clean inlet guide tube with compressed air, install dust cover on wire spool'),
('PM-R01-WT-001', 'wire_temperature', 'R-01',
 'Coolant pump impeller wear reducing flow',
 'Wire temperature rising during long welds, slower recovery between cycles',
 'Inspect coolant pump impeller for wear, replace pump if flow < 80% rated'),
('PM-R01-WT-002', 'wire_temperature', 'R-01',
 'Wire temperature sensor drift',
 'Temperature readings show gradual offset compared to contact pyrometer',
 'Recalibrate wire temperature sensor, verify with reference measurement'),
('PM-R01-WT-003', 'wire_temperature', 'R-01',
 'Coolant level low in reservoir',
 'Temperature spikes during consecutive high-duty-cycle welds',
 'Top up coolant reservoir to correct level, check for leaks in coolant circuit'),
('PM-R01-WC-001', 'welding_current', 'R-01',
 'Shielding gas nozzle misalignment',
 'Arc instability on one side of weld joint, porosity on radiograph',
 'Realign gas nozzle, check stand-off distance, verify gas coverage test'),
('PM-R01-WC-002', 'welding_current', 'R-01',
 'Wire surface contamination (rust or oil)',
 'Arc spatter increase, current oscillation, poor wetting on plate',
 'Replace wire spool, verify storage conditions, clean wire guide path'),
('PM-R01-WC-003', 'welding_current', 'R-01',
 'Earth return cable intermittent connection',
 'Intermittent current dropout during welding, voltage sensing errors',
 'Inspect earth return cable from robot base to ground grid, repair or replace as needed')
""")
print("  ✓ R-01: 10 cards inserted (4 wire_feed_speed + 3 wire_temperature + 3 welding_current)")

In [ ]:
# %% Bronze: Insert R-02 cards (10 total)
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('PM-R02-WFS-001', 'wire_feed_speed', 'R-02',
 'Wire spool damaged during handling',
 'Feed speed fluctuation, wire kinking at spool exit',
 'Replace damaged wire spool, inspect wire throughout length for kinks'),
('PM-R02-WFS-002', 'wire_feed_speed', 'R-02',
 'Feed motor bearing wear',
 'Audible noise from feed motor, speed ripple at constant setpoint',
 'Replace feed motor bearings, verify motor current signature'),
('PM-R02-WFS-003', 'wire_feed_speed', 'R-02',
 'Wire guide tube kinked near robot base',
 'Feed speed drops at specific robot arm positions',
 'Straighten or replace guide tube section, check routing path for pinch points'),
('PM-R02-WT-001', 'wire_temperature', 'R-02',
 'Coolant radiator fins clogged with dust',
 'Baseline wire temperature creeping up over weeks, slow cooldown',
 'Clean radiator fins with compressed air or water jet, check fan operation'),
('PM-R02-WT-002', 'wire_temperature', 'R-02',
 'Thermal interface compound degraded at sensor mounting',
 'Temperature reading noise and offset, erratic response to load changes',
 'Re-apply thermal compound, re-secure temperature sensor mounting'),
('PM-R02-WT-003', 'wire_temperature', 'R-02',
 'Coolant hose kinked at robot elbow',
 'Sudden temperature rise during specific arm angle, no coolant flow alarm',
 'Re-route coolant hose to avoid kinking at elbow joint, replace if damaged'),
('PM-R02-WT-004', 'wire_temperature', 'R-02',
 'Chiller unit refrigerant low',
 'Wire temperature cooling rate slowed, chiller cycling more frequently',
 'Check refrigerant pressure, recharge if low, inspect for leaks'),
('PM-R02-WC-001', 'welding_current', 'R-02',
 'Voltage sensing lead damage',
 'Current readings unstable, voltage feedback errors causing power correction',
 'Replace voltage sensing lead, verify connection at power source'),
('PM-R02-WC-002', 'welding_current', 'R-02',
 'Output rectifier diode failure',
 'Current waveform half-wave rectification, reduced average current',
 'Test output rectifier diodes, replace failed diodes in bridge assembly'),
('PM-R02-WC-003', 'welding_current', 'R-02',
 'Wire feeder clutch slipping',
 'Current drops during fast feed acceleration, inconsistent arc start',
 'Adjust clutch tension, replace clutch plate if worn beyond spec')
""")
print("  ✓ R-02: 10 cards inserted (3 wire_feed_speed + 4 wire_temperature + 3 welding_current)")

In [ ]:
# %% Bronze: Insert R-03 cards (10 total)
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('PM-R03-WFS-001', 'wire_feed_speed', 'R-03',
 'Wire diameter tolerance out of spec',
 'Feed speed oscillation at constant setpoint, poor arc starting',
 'Verify wire diameter with micrometer, replace spool if out of tolerance'),
('PM-R03-WFS-002', 'wire_feed_speed', 'R-03',
 'Feed motor controller tuning lost',
 'Speed overshoot and ringing on acceleration/deceleration ramps',
 'Re-tune PID parameters on feed motor controller, save backup settings'),
('PM-R03-WFS-003', 'wire_feed_speed', 'R-03',
 'Wire debris accumulated in feed path',
 'Incremental feed resistance increase over shift, intermittent stall events',
 'Disassemble and clean complete feed path from spool to contact tip'),
('PM-R03-WT-001', 'wire_temperature', 'R-03',
 'Temperature sensor cable shielding damaged',
 'Temperature reading noise, false drift readings',
 'Replace sensor cable with shielded type, verify grounding at both ends'),
('PM-R03-WT-002', 'wire_temperature', 'R-03',
 'Coolant flow restrictor partially blocked',
 'Wire temperature rises faster than expected during welding, slow recovery',
 'Remove and clean flow restrictor, check coolant chemistry and particle levels'),
('PM-R03-WT-003', 'wire_temperature', 'R-03',
 'Water-air heat exchanger fan failure',
 'Coolant temperature rising over shift, reduced heat rejection',
 'Replace cooling fan, clean heat exchanger fins, verify air flow direction'),
('PM-R03-WC-001', 'welding_current', 'R-03',
 'Secondary cable connection loose at power source',
 'Intermittent high-current arc strikes, voltage fluctuations',
 'Tighten all secondary cable connections to torque spec, clean contact surfaces'),
('PM-R03-WC-002', 'welding_current', 'R-03',
 'Shielding gas flow rate incorrect',
 'Arc instability with porosity, current oscillation at high frequency',
 'Verify gas flow rate at nozzle with flow meter, check for leaks in gas line'),
('PM-R03-WC-003', 'welding_current', 'R-03',
 'Welding program parameters incorrect for material thickness',
 'Current consistently above nominal for material, excessive penetration',
 'Review and correct welding program parameters (current, voltage, travel speed)'),
('PM-R03-WC-004', 'welding_current', 'R-03',
 'Primary power supply phase imbalance',
 'Current ripple at 100/120 Hz, power source derating',
 'Check incoming 3-phase power balance, contact facility maintenance if imbalance > 5%')
""")
print("  ✓ R-03: 10 cards inserted (3 wire_feed_speed + 3 wire_temperature + 4 welding_current)")

In [ ]:
# %% Bronze: Insert shared fleet cards (5 total, robot_code = NULL)
spark.sql(f"""
INSERT INTO {BRONZE_TABLE} VALUES
('PM-SHARED-001', 'wire_feed_speed', NULL,
 'Wire spool nearing end — last 10% may have surface oxidation',
 'Occasional feed speed dip, arc starting difficulty',
 'Monitor wire usage, replace spool before running out mid-weld'),
('PM-SHARED-002', 'wire_temperature', NULL,
 'Seasonal ambient temperature change affecting baseline',
 'Gradual baseline shift in wire temperature over days/weeks',
 'Review seasonal temperature compensation settings, adjust cooling parameters'),
('PM-SHARED-003', 'welding_current', NULL,
 'Contact tip age approaching end of service life',
 'Gradual current oscillation increase, deteriorating arc quality',
 'Replace contact tip per PM schedule (every 8 hours of arc-on time)'),
('PM-SHARED-004', 'wire_feed_speed', NULL,
 'Daily startup wire conditioning not performed',
 'First weld of shift shows feed speed instability',
 'Perform wire conditioning cycle at shift start: snip 50 mm, purge gas line'),
('PM-SHARED-005', 'welding_current', NULL,
 'Shielding gas cylinder replacement — composition settling',
 'Arc instability for first 2-3 welds after bottle change',
 'Purge gas line for 10 seconds after bottle change before production welding')
""")
print("  ✓ Shared fleet: 5 cards inserted")

In [ ]:
# %% Bronze: Verification — row counts by robot_code and sensor
bronze_check = spark.sql(f"""
SELECT COALESCE(robot_code, '(shared)') AS robot_code,
       sensor,
       COUNT(*) AS card_count
FROM {BRONZE_TABLE}
GROUP BY robot_code, sensor
ORDER BY robot_code, sensor
""")

total_rows = bronze_check.select(F.sum("card_count")).collect()[0][0]
print("=" * 60)
print("BRONZE LAYER — PM CARD SEED VERIFICATION")
print("=" * 60)
bronze_check.show(truncate=False)
print(f"\nTotal rows: {total_rows} ✅")
assert total_rows == 45, f"Expected 45, got {total_rows}"

---
## Phase 2: Gold Layer — Enrichment & Embeddings

In [ ]:
# %% Install embedding library on Databricks cluster
# Requires Databricks Runtime ML or standard runtime with PyPI access
%pip install sentence-transformers -q

In [ ]:
# %% Gold: Create schema and table with combined_text + embedding columns
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

# Change Data Feed must be enabled for Delta Sync with Vector Search
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    card_id       STRING,
    sensor        STRING,
    robot_code    STRING,
    cause         STRING,
    symptom       STRING,
    action        STRING,
    combined_text STRING,
    embedding     ARRAY<FLOAT>
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
)
""")

print(f"✓ Gold table ensured: {GOLD_TABLE}")
print(f"  Columns: card_id, sensor, robot_code, cause, symptom, action,")
print(f"           combined_text (STRING), embedding (ARRAY<FLOAT>)")

In [ ]:
# %% Gold: Read bronze and enrich with combined_text
bronze_df = spark.table(BRONZE_TABLE)

enriched_df = bronze_df.withColumn(
    "combined_text",
    F.concat_ws(". ",
        F.col("cause"),
        F.col("symptom"),
        F.col("action")
    )
)

print(f"Bronze rows read: {enriched_df.count()}")
enriched_df.select("card_id", "combined_text").show(5, truncate=60)
print(f"  combined_text column added ✓")

In [ ]:
# %% Gold: Compute embeddings using sentence-transformers
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {EMBEDDING_MODEL}...")
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"  Model loaded ✓ (embedding dim = {model.get_sentence_embedding_dimension()})")

# Collect combined_text to driver for batch encoding
texts = [row.combined_text for row in enriched_df.select("combined_text").collect()]
print(f"  Encoding {len(texts)} texts...")

# Batch compute embeddings
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print(f"  Embeddings computed ✓ shape={embeddings.shape}")

# Attach embeddings as a pandas column, then convert to Spark DataFrame
pdf = enriched_df.select("*").toPandas()
pdf["embedding"] = [emb.tolist() for emb in embeddings]

# Schema: all columns + embedding as array<float>
gold_schema = StructType([
    StructField("card_id", StringType(), True),
    StructField("sensor", StringType(), True),
    StructField("robot_code", StringType(), True),
    StructField("cause", StringType(), True),
    StructField("symptom", StringType(), True),
    StructField("action", StringType(), True),
    StructField("combined_text", StringType(), True),
    StructField("embedding", ArrayType(FloatType(), False), True),
])

gold_df = spark.createDataFrame(pdf, schema=gold_schema)
print(f"  Spark DataFrame created with {gold_df.count()} rows, {len(gold_df.columns)} columns ✓")

In [ ]:
# %% Gold: Write to Delta table and verify
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)

print(f"✓ Gold table written: {GOLD_TABLE}")

# Verification
gold_check = spark.sql(f"""
SELECT card_id, sensor, robot_code,
       LEFT(combined_text, 60) AS combined_text_preview,
       CARDINALITY(embedding) AS embedding_dim
FROM {GOLD_TABLE}
LIMIT 5
""")

gold_count = spark.table(GOLD_TABLE).count()
print(f"\n" + "=" * 60)
print(f"GOLD LAYER — VERIFICATION")
print(f"=" * 60)
print(f"Total rows: {gold_count}")
print(f"\nSample rows with combined_text + embedding dimension:")
gold_check.show(truncate=False)

assert gold_count == 45, f"Expected 45 rows, got {gold_count}"
print(f"✅ Gold layer seeded successfully — {gold_count} rows with {EMBEDDING_DIM}-dim embeddings")

---
## Phase 3: Databricks Vector Search Index

In [ ]:
# %% VS: Create or verify Vector Search endpoint
print(f"Connecting to Databricks workspace...")
workspace = WorkspaceClient()

try:
    endpoint = workspace.vector_search_endpoints.get_endpoint(VS_ENDPOINT_NAME)
    print(f"  Endpoint '{VS_ENDPOINT_NAME}' exists: state={endpoint.endpoint_status.state}")
    if endpoint.endpoint_status.state == "ONLINE":
        print(f"  ✓ Endpoint is ONLINE")
    else:
        print(f"  ⏳ Endpoint state = {endpoint.endpoint_status.state}, waiting for ONLINE...")
except Exception:
    print(f"  Creating endpoint '{VS_ENDPOINT_NAME}' (type=STANDARD)...")
    workspace.vector_search_endpoints.create_endpoint(
        name=VS_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    print(f"  Endpoint creation initiated. Waiting for ONLINE state...")

# Poll until ONLINE (max ~5 min)
for attempt in range(30):
    ep = workspace.vector_search_endpoints.get_endpoint(VS_ENDPOINT_NAME)
    state = ep.endpoint_status.state
    if state == "ONLINE":
        print(f"  ✓ Endpoint ONLINE (took ~{attempt * 10}s)")
        break
    elif state in ("PROVISIONING", "CREATING"):
        time.sleep(10)
    else:
        print(f"  ⚠ Unexpected endpoint state: {state}")
        break
else:
    print(f"  ⚠ Endpoint still not ONLINE after 5 min — check Databricks console")

In [ ]:
# %% VS: Create Delta Sync Vector Index on gold table
print(f"Creating Delta Sync Vector Index '{VS_INDEX_NAME}'...")
print(f"  Source table : {GOLD_TABLE}")
print(f"  Source column: combined_text (via {VS_EMBEDDING_MODEL})")
print(f"  Primary key  : card_id")

try:
    index = workspace.vector_search_indexes.create_index(
        name=VS_INDEX_NAME,
        endpoint_name=VS_ENDPOINT_NAME,
        primary_key="card_id",
        index_type=VectorSearchIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=GOLD_TABLE,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="combined_text",
                    embedding_model_endpoint_id=VS_EMBEDDING_MODEL
                )
            ]
        )
    )
    print(f"  ✓ Index '{VS_INDEX_NAME}' created")
    print(f"  Status: {index.status}")
except Exception as e:
    err_msg = str(e).lower()
    if "already exists" in err_msg:
        print(f"  ℹ Index '{VS_INDEX_NAME}' already exists")
    else:
        print(f"  ⚠ Could not create index via SDK: {e}")
        print(f"  ℹ Create it manually in Databricks UI:")
        print(f"     Catalog Explorer → {GOLD_TABLE} → Create Vector Index")
        print(f"     Endpoint  : {VS_ENDPOINT_NAME}")
        print(f"     Source col: combined_text (Model: {VS_EMBEDDING_MODEL})")
        print(f"     Key       : card_id")
        print(f"     Sync      : Triggered")

In [ ]:
# %% VS: Wait for index to become ACTIVE and test semantic query
print("Waiting for index to sync...")
for attempt in range(18):  # ~3 minutes
    try:
        idx = workspace.vector_search_indexes.get_index(VS_INDEX_NAME)
        status = idx.status
        print(f"  Attempt {attempt+1}: index_status={status}")
        if status and status.ready:
            print(f"  ✓ Index is READY — proceeding to test query")
            break
    except Exception as e:
        print(f"  Attempt {attempt+1}: index not ready yet — {e}")
    time.sleep(10)
else:
    print(f"  ⚠ Index not ready after ~3 min. Continue to test query anyway...")

print(f"\n" + "=" * 60)
print("SEMANTIC QUERY TEST")
print("=" * 60)

# Test queries using query_text — Databricks VS endpoint computes the
# embedding server-side using the same model (databricks-bge-large-en)
# that was configured on the index, ensuring dimensional alignment.
test_queries = [
    "wire feed speed jam and debris",
    "temperature rising during welding",
    "unstable arc and current oscillation",
]

success_count = 0
for q in test_queries:
    print(f"\nQuery: \"{q}\"")
    try:
        result = workspace.vector_search_indexes.query_index(
            index_name=VS_INDEX_NAME,
            columns=["card_id", "sensor", "robot_code", "combined_text"],
            query_text=q,
            num_results=3,
        )
        if result.result and result.result.data_array:
            for row in result.result.data_array:
                print(f"    → {row[0]} | sensor={row[1]} | robot={row[2]} | {row[3][:60]}...")
            success_count += 1
        else:
            print(f"    (no results)")
    except Exception as e:
        print(f"    ⚠ Query failed: {e}")

print(f"\n✅ {success_count}/{len(test_queries)} semantic queries completed successfully!")

---
## Summary &amp; Next Steps

### Created Resources

| Layer  | Resource | Location |
|--------|----------|----------|
| Bronze | Delta table (45 raw rows) | `demo.bronze_layer.pm_cards` |
| Gold   | Delta table with embeddings | `demo.gold_layer.pm_cards` |
| Search | Vector Search endpoint | `pm-cards-vs-endpoint` |
| Search | Delta Sync Vector Index | `demo.gold_layer.pm_cards_vs_index` |

### Gold Table Columns

| Column | Type | Description |
|--------|------|-------------|
| `card_id` | STRING | Primary key |
| `sensor` | STRING | Sensor type (wire_feed_speed, wire_temperature, welding_current) |
| `robot_code` | STRING | Robot identifier (R-00 to R-03, or NULL for shared) |
| `cause` | STRING | Root cause description |
| `symptom` | STRING | Observable symptom |
| `action` | STRING | Corrective / preventive action |
| `combined_text` | STRING | `cause. symptom. action` — used by Vector Search for embedding |
| `embedding` | ARRAY&lt;FLOAT&gt; | Pre-computed 384-dim embedding (for ChromaDB/local fallback) |

### How the Pieces Fit Together

```
Bronze (raw)  ──→  Gold (enriched)  ──→  Vector Search Index
  raw cards         + combined_text        Delta Sync on combined_text
  6 columns         + embedding (384d)     Model: databricks-bge-large-en
                    + CDF enabled          Sync: Triggered
```

- The **Vector Search index** uses Databricks' `databricks-bge-large-en` model endpoint to compute embeddings from `combined_text` server-side. Queries use `query_text` — Databricks handles embedding computation automatically.
- The **pre-computed `embedding` column** (384-dim, `all-MiniLM-L6-v2`) is available for local/offline use such as the ChromaDB fallback in `backend/app/robot_agent/agent/rag_tools.py`.

### Next Steps for the Backend Agent

Update `backend/app/robot_agent/agent/rag_tools.py` to add a third query path using Vector Search (in addition to the existing Databricks ILIKE and ChromaDB fallback):

```python
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()
result = workspace.vector_search_indexes.query_index(
    index_name="demo.gold_layer.pm_cards_vs_index",
    columns=["card_id", "sensor", "robot_code", "cause", "symptom", "action"],
    query_text="&lt;user query&gt;",
    num_results=5,
)
```

**Source of truth:** `anomaly-detection/data/sql/seed_databricks_pm_cards.sql`